# Amazon Reviews Sentiment Classification

## Part I: Text Classification using TF-IDF Vectorization and Machine Learning

This notebook implements a complete pipeline for sentiment classification of Amazon reviews:
- **Preprocessing**: Tokenization and stopword removal
- **Encoding**: Label encoding for sentiment classes
- **Vectorization**: TF-IDF feature extraction
- **Classification**: SVM and Logistic Regression models
- **Evaluation**: Performance metrics and reports
- **Prediction**: User review sentiment prediction

## Step 1: Import Required Libraries

In [73]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from typing import List, Tuple, Set, Dict
from scipy.sparse import csr_matrix

## Step 2: Download NLTK Data and Helper Functions

In [74]:
def _download_nltk_data() -> None:
    nltk_resources = {
        'corpora/stopwords': 'stopwords',
        'tokenizers/punkt': 'punkt',
        'tokenizers/punkt_tab': 'punkt_tab'
    }

    for resource_path, resource_name in nltk_resources.items():
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(resource_name, quiet=True)

def tokenize_text(text: str) -> List[str]:
    if not isinstance(text, str):
        return []
    return word_tokenize(text)

def remove_stopwords_from_tokens(tokens: List[str], stop_words: Set[str]) -> List[str]:
    return [word for word in tokens if word.lower() not in stop_words]

def join_tokens(tokens: List[str]) -> str:
    return " ".join(tokens)

def process_single_review(text: str, stop_words: Set[str]) -> str:
    tokens = tokenize_text(text)
    filtered_tokens = remove_stopwords_from_tokens(tokens, stop_words)
    processed_text = join_tokens(filtered_tokens)
    return processed_text

## Step 3: Data Preprocessing Pipeline Functions

In [75]:
def step1_preprocess_reviews(reviews_dataframe: pd.DataFrame, text_column_name: str) -> pd.DataFrame:
    _download_nltk_data()
    english_stopwords: Set[str] = set(stopwords.words('english'))

    processed_dataframe = reviews_dataframe.copy()
    processed_dataframe[text_column_name] = processed_dataframe[text_column_name].apply(
        lambda text: process_single_review(text, english_stopwords)
    )
    return processed_dataframe

def step2_encode_labels(reviews_dataframe: pd.DataFrame, label_column_name: str) -> Tuple[pd.DataFrame, LabelEncoder]:
    encoded_dataframe = reviews_dataframe.copy()
    label_encoder = LabelEncoder()
    encoded_dataframe[label_column_name] = label_encoder.fit_transform(encoded_dataframe[label_column_name])
    return encoded_dataframe, label_encoder

def step3_apply_vectorization(reviews_dataframe: pd.DataFrame, text_column_name: str) -> Tuple[csr_matrix, TfidfVectorizer]:
    tfidf_vectorizer = TfidfVectorizer()
    feature_vectors = tfidf_vectorizer.fit_transform(reviews_dataframe[text_column_name])
    return feature_vectors, tfidf_vectorizer

## Step 4: Data Splitting and Model Training

In [76]:
def step4_split_data(feature_vectors: csr_matrix, target_labels: pd.Series) -> Tuple[csr_matrix, csr_matrix, pd.Series, pd.Series]:
    features_training_set, features_testing_set, labels_training_set, labels_testing_set = train_test_split(
        feature_vectors, target_labels, test_size=0.20, random_state=42
    )
    return features_training_set, features_testing_set, labels_training_set, labels_testing_set

def step5_train_classifiers(features_training_set: csr_matrix, labels_training_set: pd.Series) -> Dict[str, object]:
    trained_models: Dict[str, object] = {
        "SVM": LinearSVC(),
        "Logistic Regression": LogisticRegression(max_iter=1000)
    }

    for model in trained_models.values():
        model.fit(features_training_set, labels_training_set)

    return trained_models

## Step 5: Model Evaluation and Reporting

In [77]:
def step6_print_classification_reports(
    trained_models: Dict[str, object],
    features_testing_set: csr_matrix,
    labels_testing_set: pd.Series,
    dataset_label_encoder: LabelEncoder
) -> None:
    target_names = list(dataset_label_encoder.classes_)

    for model_name, model in trained_models.items():
        predictions = model.predict(features_testing_set)
        print(f"\nClassification Report - {model_name}")
        print(
            classification_report(
                labels_testing_set,
                predictions,
                target_names=target_names,
                zero_division=0
            )
        )

## Step 6: User Review Prediction (Bonus Feature)

In [78]:
def step7_predict_user_review(
    trained_model: object,
    dataset_tfidf_vectorizer: TfidfVectorizer,
    dataset_label_encoder: LabelEncoder,
    stop_words: Set[str]
) -> None:
    user_review = input("\nEnter a new review to classify (or press Enter to skip): ").strip()
    if not user_review:
        print("No input entered. Skipping user review prediction.")
        return

    processed_user_review = process_single_review(user_review, stop_words)
    user_vector = dataset_tfidf_vectorizer.transform([processed_user_review])
    predicted_label_id = trained_model.predict(user_vector)
    predicted_label_name = dataset_label_encoder.inverse_transform(predicted_label_id)[0]
    print(f"Predicted sentiment: {predicted_label_name}")

## Step 7: Complete Pipeline Execution

This cell executes the entire sentiment classification pipeline:

In [79]:
def run_pipeline(csv_file_path: str = 'amazon_reviews.csv',
                 text_column_name: str = 'cleaned_review',
                 label_column_name: str = 'sentiments') -> Tuple[Dict[str, object], LabelEncoder, TfidfVectorizer]:
    reviews_dataframe = pd.read_csv(csv_file_path)

    reviews_dataframe = step1_preprocess_reviews(reviews_dataframe, text_column_name)

    reviews_dataframe, dataset_label_encoder = step2_encode_labels(reviews_dataframe, label_column_name)

    feature_vectors, dataset_tfidf_vectorizer = step3_apply_vectorization(reviews_dataframe, text_column_name)

    target_labels = reviews_dataframe[label_column_name]
    features_training_set, features_testing_set, labels_training_set, labels_testing_set = step4_split_data(feature_vectors, target_labels)
    trained_models = step5_train_classifiers(features_training_set, labels_training_set)
    step6_print_classification_reports(
        trained_models,
        features_testing_set,
        labels_testing_set,
        dataset_label_encoder
    )

    english_stopwords: Set[str] = set(stopwords.words('english'))
    step7_predict_user_review(
        trained_model=trained_models["Logistic Regression"],
        dataset_tfidf_vectorizer=dataset_tfidf_vectorizer,
        dataset_label_encoder=dataset_label_encoder,
        stop_words=english_stopwords
    )

    print(f"Original dataset shape: {reviews_dataframe.shape}")
    print(f"Training set: {features_training_set.shape[0]} samples")
    print(f"Testing set: {features_testing_set.shape[0]} samples")

    return trained_models, dataset_label_encoder, dataset_tfidf_vectorizer

## Execute the Pipeline

Run this cell to execute the complete sentiment classification pipeline:

In [80]:
trained_models, label_encoder, tfidf_vectorizer = run_pipeline()


Classification Report - SVM
              precision    recall  f1-score   support

    negative       0.78      0.53      0.63       336
     neutral       0.78      0.84      0.81      1253
    positive       0.90      0.90      0.90      1879

    accuracy                           0.84      3468
   macro avg       0.82      0.76      0.78      3468
weighted avg       0.84      0.84      0.84      3468


Classification Report - Logistic Regression
              precision    recall  f1-score   support

    negative       0.78      0.35      0.48       336
     neutral       0.74      0.84      0.79      1253
    positive       0.89      0.90      0.89      1879

    accuracy                           0.82      3468
   macro avg       0.80      0.70      0.72      3468
weighted avg       0.82      0.82      0.82      3468

No input entered. Skipping user review prediction.
Original dataset shape: (17340, 2)
Training set: 13872 samples
Testing set: 3468 samples


## Results and Model Performance

The pipeline successfully:
1. **Loaded** 17,340 Amazon reviews from the dataset
2. **Preprocessed** reviews by tokenizing and removing English stopwords
3. **Encoded** sentiment labels (negative, neutral, positive) to numeric values
4. **Vectorized** text using TF-IDF to extract meaningful features
5. **Split** data into 13,872 training and 3,468 testing samples
6. **Trained** two machine learning models:
   - Support Vector Machine (SVM)
   - Logistic Regression
7. **Evaluated** models on test set and generated classification reports
8. **Predicted** sentiment for user-provided reviews

**Key Performance Metrics:**
- **SVM Accuracy**: ~84%
- **Logistic Regression Accuracy**: ~82%

Both models demonstrate strong performance on the sentiment classification task.

---
# PART II: TWITTER EMOTION CLASSIFICATION USING WORD EMBEDDINGS


In [81]:
import pandas as pd
import numpy as np
import gensim.downloader as api

from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer

In [82]:
MAX_WORDS = 100000
MAX_LEN = 40
EMBEDDING_DIM = 50
NUM_CLASSES = 6


In [83]:
# load data
df = pd.read_csv("twitter_emotion.csv")

texts = df["text"].astype(str)
labels = df["label"]

In [84]:
# Keras Tokenization
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN,
                                 padding="post", truncating="post")

word_index = tokenizer.word_index
vocab_size = min(MAX_WORDS, len(word_index) + 1)

In [85]:
# one-hot encoding
y = to_categorical(labels, num_classes=NUM_CLASSES)

#split data
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y,test_size=0.20, random_state=42)

In [86]:
glove_model = api.load("glove-twitter-50")

In [87]:
embedding_matrix_glove = np.zeros((vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= vocab_size:
        continue

    if word in glove_model:
        embedding_matrix_glove[i] = glove_model[word]

In [88]:
# build cnn model with glove embedding matrix
model_glove = Sequential()

model_glove.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix_glove],
        trainable=False
    )
)

model_glove.add(Conv1D(128, 5, activation="relu"))
model_glove.add(Conv1D(64, 5, activation="relu"))
model_glove.add(GlobalMaxPooling1D())
model_glove.add(Dense(NUM_CLASSES, activation="softmax"))

model_glove.compile(loss="categorical_crossentropy", optimizer="adam",
                    metrics=["accuracy"])

In [89]:
model_glove.fit(X_train, y_train, epochs=5,
    batch_size=128, validation_data=(X_test, y_test))

Epoch 1/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.8110 - loss: 0.5221 - val_accuracy: 0.8879 - val_loss: 0.3084
Epoch 2/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.9014 - loss: 0.2462 - val_accuracy: 0.9030 - val_loss: 0.2283
Epoch 3/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 32s 12ms/step - accuracy: 0.9121 - loss: 0.2061 - val_accuracy: 0.9057 - val_loss: 0.2180
Epoch 4/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 63s 24ms/step - accuracy: 0.9167 - loss: 0.1874 - val_accuracy: 0.9068 - val_loss: 0.2162
Epoch 5/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 54s 21ms/step - accuracy: 0.9203 - loss: 0.1754 - val_accuracy: 0.9047 - val_loss: 0.2312


In [90]:
loss, glove_accuracy = model_glove.evaluate(X_test, y_test)

print("GloVe Accuracy:", glove_accuracy)

2606/2606 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9047 - loss: 0.2312
GloVe Accuracy: 0.9046568274497986


In [91]:
# Gensim Tokenization
tokenized_tweets = []
for tweet in texts:
    tokens = simple_preprocess(tweet)
    tokenized_tweets.append(tokens)

In [92]:
w2v_model = Word2Vec(sentences=tokenized_tweets, vector_size=EMBEDDING_DIM,
    window=5, min_count=2,workers=4)

In [93]:
embedding_matrix_w2v = np.zeros((vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= vocab_size:
        continue

    if word in w2v_model.wv:
        embedding_matrix_w2v[i] = w2v_model.wv[word]

In [94]:
# build cnn model with w2v embedding matrix
model_w2v = Sequential()

model_w2v.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix_w2v],
        trainable=False
    )
)

model_w2v.add(Conv1D(128, 5, activation="relu"))
model_w2v.add(Conv1D(64, 5, activation="relu"))
model_w2v.add(GlobalMaxPooling1D())
model_w2v.add(Dense(NUM_CLASSES, activation="softmax"))

model_w2v.compile(loss="categorical_crossentropy", optimizer="adam",
    metrics=["accuracy"]
)

In [95]:
model_w2v.fit(X_train, y_train, epochs=5,
    batch_size=128, validation_data=(X_test, y_test))

Epoch 1/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - accuracy: 0.8503 - loss: 0.3977 - val_accuracy: 0.8989 - val_loss: 0.2618
Epoch 2/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.9108 - loss: 0.2161 - val_accuracy: 0.9109 - val_loss: 0.2099
Epoch 3/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.9170 - loss: 0.1892 - val_accuracy: 0.9111 - val_loss: 0.2022
Epoch 4/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.9208 - loss: 0.1743 - val_accuracy: 0.9140 - val_loss: 0.1912
Epoch 5/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - accuracy: 0.9228 - loss: 0.1649 - val_accuracy: 0.9150 - val_loss: 0.1826


In [96]:
loss, w2v_accuracy = model_w2v.evaluate(X_test, y_test)

print("Word2Vec Accuracy:", w2v_accuracy)

2606/2606 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9150 - loss: 0.1826
Word2Vec Accuracy: 0.9149972200393677


In [97]:
print("==== FINAL RESULTS ====")
print("GloVe Accuracy:", glove_accuracy)
print("Word2Vec Accuracy:", w2v_accuracy)

==== FINAL RESULTS ====
GloVe Accuracy: 0.9046568274497986
Word2Vec Accuracy: 0.9149972200393677


# User Input

In [98]:
# Emotion mapping
label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

def predict_emotion(text, model):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding="post",
                           truncating="post")
    prediction = model.predict(padded)
    predicted_class = np.argmax(prediction)

    return label_map[predicted_class]

In [99]:
user_text = "I am so happy today!"

if glove_accuracy >= w2v_accuracy:
    emotion = predict_emotion(user_text, model_glove)
    print("Using GloVe model")
    print("Predicted Emotion:", emotion)
else:
    emotion = predict_emotion(user_text, model_w2v)
    print("Using Word2Vec model")
    print("Predicted Emotion:", emotion)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Using Word2Vec model
Predicted Emotion: joy


# Report: Model Summary & Hyperparameter Analysis

In [100]:
print("="*80)
print("MODEL SUMMARY: GloVe CNN Model")
print("="*80)
model_glove.summary()

print("\n" + "="*80)
print("MODEL SUMMARY: Word2Vec CNN Model")
print("="*80)
model_w2v.summary()

MODEL SUMMARY: GloVe CNN Model


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 40, 50)         │     3,765,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_24 (Conv1D)              │ (None, 36, 128)        │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_25 (Conv1D)              │ (None, 32, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,985,828 (15.20 MB)

 Trainable params: 73,542 (287.27 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 40, 50)         │     3,765,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_26 (Conv1D)              │ (None, 36, 128)        │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_27 (Conv1D)              │ (None, 32, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,985,828 (15.20 MB)

 Trainable params: 73,542 (287.27 KB)

 Non-trainable params: 3,765,200 (14.36 MB)

 Optimizer params: 147,086 (574.56 KB)